# 1.11 很多參數各自該往哪邊動？


兩個旋鈕目前是 `[1,2]`。第一個 w0 希望靠近 3，第二個 w1 希望靠近 0，但第二個誤差只計半份。代價是 `(w0-3)²+0.5*w1²`，目前兩項為 4 與 2，總代價 6。

我們希望拿到每格自己的調整方向。只微動一格、其餘不動，得到的敏感度叫**偏導數**。沿用[1.9](https://birdhackor.github.io/tiny-perceptron-vlm/1.9.html)的平方距離算例：距離若是 d，微增 h 時，代價改變為 `(d+h)²-d²=2*d*h+h²`；除以步子 h，得到 `2*d+h`。h 越接近零，這個比例越接近 `2*d`，所以平方距離的敏感度是目前距離的兩倍。代價前若有常數權重，也要把敏感度乘上同一權重。

第一格距離是 `1-3=-2`，敏感度為 `2*(1-3)=-4`。第二格距離是 `2-0=2`，但代價只計半份，敏感度為 `0.5*2*2=2`；按參數位置排成梯度 `[-4,2]`。

| 參數 | 目前值 | 梯度 | 希望降低代價時的小步方向 |
| --- | ---: | ---: | --- |
| w0 | 1 | -4 | 增加 |
| w1 | 2 | +2 | 減少 |


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch

w = torch.tensor([1.0, 2.0], requires_grad=True)
first_cost = (w[0] - 3).square()
second_cost = 0.5 * w[1].square()
loss = first_cost + second_cost
loss.backward()
print("參數", w.detach())
print("各項與總代價", first_cost.item(), second_cost.item(), loss.item())
print("梯度", w.grad)
assert torch.allclose(w.grad, torch.tensor([-4.0, 2.0]))

參數 tensor([1., 2.])
各項與總代價 4.0 2.0 6.0
梯度 tensor([-4.,  2.])


程式先分別算兩項再加總，列印可核對 4、2、6。對總代價做 `backward()` 後，參數仍 `[1,2]`，`.grad` 是 `[-4,2]`。它與參數形狀相同，每格敏感度都有自己的參數可對應。

為什麼一個總數可得出不同方向？計算圖記住 w0 影響第一項，w1 影響第二項；求導沿各自的路往回算。總代價 6 本身沒有這些方向資訊，不能拿 6 當成每格共同的更新量。

梯度也不是新參數。不能直接令 `w=[-4,2]`；我們會沿梯度的反方向走一小步，下一節再決定步子的比例。

練習只把第二項係數 0.5 改成 2，預期總代價變 12、梯度變 `[-4,8]`。將斷言的第二格同步改成 8 再核對：第一格沒變，第二項的重要程度增加四倍，其梯度也增加四倍。

<details>
<summary>補充與重做</summary>

這裡的 w 是直接建立的可調來源，常叫葉節點。通常在這些參數的 `.grad` 讀更新所需梯度，中間結果的 `.grad` 不一定自動儲存。`allclose` 作逐格近似比較，`assert` 只檢查，不調參數。

</details>
